# ML_LR_comparison — Colab Runner

動態學習率排程對比 + Grad-CAM 視覺化。支援三組實驗：
- `tiny_imagenet` (200 類，pretrained，20 epoch)
- `imagewoof` (10 類細粒度狗品種，pretrained，20 epoch)
- `imagewoof_scratch` (10 類，**from scratch + MixUp + RandAugment**，80 epoch)

**前置：** Runtime → Change runtime type → 選 GPU (T4 / V100 / A100 皆支援，會自動套對應 profile)。

**Demo 模式**（推薦給只想看 Grad-CAM GIF 的場景）：cell 4 設 `DEMO_MODE = True`，自動縮 epoch 並寫到 `experiments_demo/`（不覆蓋既有 `experiments/`）。

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 掛載 Google Drive（用來持久化 logs / checkpoints）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ML_LR_comparison'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)

## 3. Clone 本 repo + 安裝套件

In [ ]:
%cd /content
!rm -rf ML_LR_comparison
!git clone https://github.com/eric20041027/ML_LR_comparison.git
%cd /content/ML_LR_comparison
!pip install -q -r requirements.txt

## 4. 選擇實驗組 + 共用參數

**`DEMO_MODE = True`** → epoch 數大幅縮減（只為產 Grad-CAM GIF）+ 輸出寫到 `experiments_demo/` 不覆蓋既有結果。

**T4 demo 時間預估（每 dataset 跑一次）：**
- `imagewoof` (10 ep)：~3 分/組 × 5 ≈ **15 分鐘**
- `imagewoof_scratch` (25 ep)：~7 分/組 × 5 ≈ **35 分鐘**
- `tiny_imagenet` (8 ep)：~15 分/組 × 5 ≈ **75 分鐘**

三組 demo 總計 ~2 小時。若想最快 demo，建議**只跑 `imagewoof` 和 `imagewoof_scratch` (~50 分鐘)** — 這兩個的 Grad-CAM 在狗臉上聚焦，視覺效果最強。

In [ ]:
import torch

# === 切換這裡 ===
DATASET = 'imagewoof_scratch'   # 'tiny_imagenet' / 'imagewoof' / 'imagewoof_scratch'
DEMO_MODE = True                # True = 縮短 epoch + 寫到 experiments_demo/

# Demo 模式下的 epoch 預算 — 為 Grad-CAM GIF 演進設計，非為了 acc 數字
DEMO_EPOCHS = {
    'tiny_imagenet': 8,
    'imagewoof': 10,
    'imagewoof_scratch': 25,
}
FULL_EPOCHS = {
    'tiny_imagenet': 20,
    'imagewoof': 20,
    'imagewoof_scratch': 80,
}

# Map config-group -> actual dataset folder for the downloader
DATA_DATASET = 'imagewoof' if DATASET.startswith('imagewoof') else DATASET

EPOCHS = DEMO_EPOCHS[DATASET] if DEMO_MODE else FULL_EPOCHS[DATASET]
DATA_DIR = '/content/data'
EXP_SUBDIR = 'experiments_demo' if DEMO_MODE else 'experiments'
OUTPUT_DIR = f'{DRIVE_ROOT}/{EXP_SUBDIR}/{DATASET}'

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
PROFILE = 'a100' if 'A100' in gpu_name else 't4'
print(f'Mode         : {"DEMO" if DEMO_MODE else "FULL"}')
print(f'Group        : {DATASET}')
print(f'Real dataset : {DATA_DATASET}')
print(f'Epochs       : {EPOCHS}')
print(f'Detected GPU : {gpu_name}  ->  profile = {PROFILE}')
print(f'Output dir   : {OUTPUT_DIR}')

## 5. 下載資料集（首次需要；已下載過會跳過）

In [ ]:
!python -m scripts.download_data --dataset {DATA_DATASET} --data-dir {DATA_DIR}

## 6. 跑 5 組實驗（含每 epoch Grad-CAM 紀錄）

`--gradcam-per-epoch` 會在每 epoch 末對一張固定 val 圖計算 Grad-CAM，存 PNG 到 `cam_per_epoch/`，供第 9 節合成 GIF。每張 PNG ~50KB，overhead < 1 秒/epoch。

In [ ]:
!python -m scripts.run_all \
    --dataset {DATASET} \
    --data-root {DATA_DIR} \
    --output-dir {OUTPUT_DIR} \
    --epochs {EPOCHS} \
    --profile {PROFILE} \
    --gradcam-per-epoch

## 7. 量化視覺化：LR 曲線 + Loss/Acc 曲線

In [ ]:
!python -m src.plot_lr --experiments-dir {OUTPUT_DIR} --out {OUTPUT_DIR}/lr_curves.png
!python -m src.plot_curves --experiments-dir {OUTPUT_DIR} --out {OUTPUT_DIR}/curves.png

from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/lr_curves.png'))
display(Image(f'{OUTPUT_DIR}/curves.png'))

## 8. 質化視覺化：Grad-CAM 對比圖（5 排程 × 早/中/晚期，3 個 checkpoint snapshot）

In [ ]:
!python -m src.gradcam_viz \
    --experiments-dir {OUTPUT_DIR} \
    --data-root {DATA_DIR} \
    --out {OUTPUT_DIR}/grad_cam_grid.png

from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/grad_cam_grid.png'))

## 9. 質化視覺化（動畫）：每 epoch Grad-CAM GIF

把第 6 節記錄的每 epoch PNG 合成一個動畫 GIF，方便看「注意力如何隨 epoch 演進」。

輸出：`cam_evolution.gif`（GitHub README 可直接渲染）。

In [ ]:
# total_duration_ms: 控制 GIF 總長度（自動算每 frame 多少毫秒）
# 短訓練設 4 秒；長訓練 (25+ ep) 設 6 秒
GIF_TOTAL_MS = 6000 if EPOCHS >= 20 else 4000

!python -m scripts.make_cam_gif \
    --experiments-dir {OUTPUT_DIR} \
    --out {OUTPUT_DIR}/cam_evolution.gif \
    --target-total-ms {GIF_TOTAL_MS}

from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/cam_evolution.gif'))

## 10. (可選) 在 Colab 內看 TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR